# Natural Gas Price Estimation

JPMorgan Chase Quantitative Research — Forage Simulation

This notebook analyzes monthly natural gas prices (Oct 2020 – Sep 2024), identifies seasonal patterns, and builds a model to estimate prices for any historical date and extrapolate one year into the future.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = Path("Natural Gas Data.csv")

## 1. Load and Explore the Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = ["date", "price"]
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%y")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

print(f"Observations: {len(df)}")
print(f"Date range:   {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Price range:  ${df['price'].min():.2f} – ${df['price'].max():.2f}")
df.head()

## 2. Visualize the Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["date"], df["price"], marker="o", markersize=4, linewidth=1.5, color="#1f4e79")
ax.set_title("Monthly Natural Gas Prices (Month-End)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.set_ylim(bottom=8)
plt.tight_layout()
plt.show()

## 3. Seasonal Analysis

Natural gas prices typically rise in winter (heating demand) and soften in late spring / early summer.

In [ ]:
df["month"] = df["date"].dt.month
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

seasonal_avg = df.groupby("month")["price"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 13), seasonal_avg.values, color="#2e75b6", edgecolor="white")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names)
axes[0].set_title("Average Price by Calendar Month", fontweight="bold")
axes[0].set_ylabel("Average Price ($)")

df.boxplot(column="price", by="month", ax=axes[1])
axes[1].set_title("Price Distribution by Month", fontweight="bold")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Price ($)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. Build the Pricing Model

**Approach:**
- **Trend:** linear regression on time (captures the upward drift)
- **Seasonality:** average monthly deviation from trend (Jan–Dec factors)
- **Historical dates:** linear interpolation between month-end model values
- **Future dates (up to 1 year):** extend the trend and apply the same seasonal pattern

In [ ]:
from natural_gas_model import NaturalGasPriceModel, get_model

model = get_model()

print("Model fitted.")
print(f"  Trend:     {model.trend_intercept:.3f} + {model.trend_slope:.6f} × days")
print(f"  Last obs:  {model.last_date.date()}")
print(f"  Forecast:  through {model.forecast_end.date()}")
print("\nSeasonal factors ($ deviation from trend):")
for m in range(1, 13):
    sign = "+" if model.seasonal_factors[m] >= 0 else ""
    print(f"  {month_names[m-1]:>3}: {sign}{model.seasonal_factors[m]:.3f}")

## 5. Decompose into Trend + Seasonality

In [ ]:
df["trend"] = df["date"].apply(
    lambda d: model.trend_slope * model._days_from_start(d) + model.trend_intercept
)
df["seasonal"] = df["date"].dt.month.map(model.seasonal_factors)
df["residual"] = df["price"] - df["trend"] - df["seasonal"]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(df["date"], df["price"], color="#1f4e79")
axes[0].set_ylabel("Observed")
axes[0].set_title("Additive Decomposition: Price = Trend + Seasonal + Residual", fontweight="bold")
axes[1].plot(df["date"], df["trend"], color="#2e75b6")
axes[1].set_ylabel("Trend")
axes[2].plot(df["date"], df["seasonal"], color="#c55a11")
axes[2].set_ylabel("Seasonal")
axes[3].plot(df["date"], df["residual"], color="#548235")
axes[3].set_ylabel("Residual")
axes[3].set_xlabel("Date")
plt.tight_layout()
plt.show()